Analyzes microscopy images to quantify fluorescence intensities inside and outside lipids

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.filters import (
    gaussian, threshold_otsu, threshold_multiotsu, unsharp_mask
)
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, binary_dilation, disk
from skimage import exposure, io
from cellpose import models
import czifile

model = models.Cellpose(model_type='cyto')

In [2]:
COLOR_SCHEME = 'viridis'

In [ ]:
def show_image(image, cmap=COLOR_SCHEME):
    """Display a single image with a colorbar."""
    plt.imshow(image, cmap=cmap)
    plt.colorbar()
    plt.show()

def show_images(images, titles=None, cmap=COLOR_SCHEME):
    """Display multiple images in a grid with colorbars."""
    n = len(images)
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))

    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for i, img in enumerate(images):
        ax = axes[i // cols][i % cols]
        im = ax.imshow(img, cmap=cmap)
        if titles:
            ax.set_title(titles[i])
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for j in range(i + 1, rows * cols):
        ax = axes[j // cols][j % cols]
        ax.axis('off')

    plt.tight_layout()
    plt.show()

def calculate_average_intensity(image):
    """Calculate the average intensity of the image."""
    return np.mean(image)

def find_lipid_mask(image):
    """Find the lipid mask using otsu thresholding."""
    threshold = threshold_otsu(image)
    lipid_mask = (image > threshold).astype(int)
    
    return lipid_mask

In [ ]:
def analyze_image(image_path, basename, mask_channel='green'):
    """
    Analyze a single image and return a DataFrame with intensity metrics.
    image_path: Path to the image file.
    basename: Base name of the image file for identification.
    mask_channel: Channel to use for segmenting lipid mask ('orange' or 'green').
    """

    # Load image
    image = czifile.imread(image_path)
    image_squeezed = np.squeeze(image)

    # Select mask channel
    if mask_channel == 'orange':
        mask_image = image_squeezed[0]
    else:
        mask_image = image_squeezed[1]

    mask = find_lipid_mask(mask_image)
    inverse_mask = 1 - mask

    intensities_whole_image = []
    intensities_without_lipids = []
    intensities_only_lipids = []

    show_images([image_squeezed[0], image_squeezed[1], image_squeezed[2], mask], titles=['Orange Channel', 'Green Channel', 'Red Channel', 'Lipid Mask'])

    for c in range(image_squeezed.shape[0]):
        channel = image_squeezed[c]

        intensities_whole_image.append(calculate_average_intensity(channel))
        intensities_without_lipids.append(calculate_average_intensity(channel[inverse_mask == 1]))
        intensities_only_lipids.append(calculate_average_intensity(channel[mask == 1]))

    df = pd.DataFrame({
        "File_Name": [basename],
        "Orange_Average_Intensity_Whole_Image": [intensities_whole_image[0]],
        "Orange_Intensity_Without_Lipids": [intensities_without_lipids[0]],
        "Orange_Intensity_Only_Lipids": [intensities_only_lipids[0]],
        "Green_Average_Intensity_Whole_Image": [intensities_whole_image[1]],
        "Green_Intensity_Without_Lipids": [intensities_without_lipids[1]],
        "Green_Intensity_Only_Lipids": [intensities_only_lipids[1]],
        "Red_Average_Intensity_Whole_Image": [intensities_whole_image[2]],
        "Red_Intensity_Without_Lipids": [intensities_without_lipids[2]],
        "Red_Intensity_Only_Lipids": [intensities_only_lipids[2]],
        "Orange_to_Green_Ratio_Whole_Image": [intensities_whole_image[0] / intensities_whole_image[1]],
        "Orange_to_Green_Ratio_Without_Lipids": [intensities_without_lipids[0] / intensities_without_lipids[1]],
        "Orange_to_Green_Ratio_Only_Lipids": [intensities_only_lipids[0] / intensities_only_lipids[1]],
    })

    return df

In [5]:
def analyze_all_images(image_folder):
    all_data_green = []
    all_data_orange = []

    for filename in os.listdir(image_folder):
        if filename.lower().endswith(".czi"):
            print(filename)

            path = os.path.join(image_folder, filename)
            basename = os.path.splitext(filename)[0]

            df_green = analyze_image(path, basename, 'green')
            df_orange = analyze_image(path, basename, 'orange')

            all_data_green.append(df_green)
            all_data_orange.append(df_orange)

            print("-" * 100)

    base_name = image_folder.replace("/", "_")
    pd.concat(all_data_green, ignore_index=True).to_excel(f"{base_name}_analysis_green_mask.xlsx", index=False)
    pd.concat(all_data_orange, ignore_index=True).to_excel(f"{base_name}_analysis_orange_mask.xlsx", index=False)


def analyze_all_subfolders(parent_folder):
    for subfolder in os.listdir(parent_folder):
        full_path = os.path.join(parent_folder, subfolder)
        if os.path.isdir(full_path):
            analyze_all_images(full_path)

In [6]:
# folders = ['012025_FRET_optimization/new_settings','012725_FRET_optimize/C12_settings','012725_FRET_optimize/Lipidtox_settings']
# folders = ['021725_FRET_Optimization/490_550','021725_FRET_Optimization/490_600','021725_FRET_Optimization/500_550']
folder = 'Lipid peroxidation FRET 4725'

In [ ]:
analyze_all_images(folder)
